# IMDB Dataset Exploration

Load the Kaggle *Bag of Words Meets Bags of Popcorn* data using the project's shared configuration and helpers, then establish a basic understanding of its quality and structure before modeling.

## Imports

In [2]:
# standard library
from pathlib import Path
import sys

# third-party
import pandas as pd
from IPython.display import display

# local
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

scripts_path = str(PROJECT_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from imdb_dataset_helpers import (
    build_cleanup_report,
    build_dataset_overview,
    clean_reviews,
    duplicate_review_examples,
    load_datasets,
    print_summary,
)

## Acquire and load the data

The shared helper downloads the Kaggle competition bundle only when the configured files are absent.

In [3]:
train, test = load_datasets()

Kaggle files already present; using: (project_root)/data/raw


In [4]:
print_summary("Labeled training data", train)
print_summary("Test data", test)


Labeled training data
Shape: (25000, 3)
Columns: ['id', 'sentiment', 'review']
Sample reviews:
  0: sentiment=1 With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watch...
  1: sentiment=1 \The Classic War of the Worlds\" by Timothy Hines is a very entertaining film that obviously goes to great effort and lengths to faithfully recreate H. G. Wells...

Test data
Shape: (25000, 2)
Columns: ['id', 'review']
Sample reviews:
  0: Naturally in a film who's main themes are of mortality, nostalgia, and loss of innocence it is perhaps not surprising that it is rated more highly by older view...
  1: This movie is a disaster within a disaster film. It is full of great action scenes, which are only meaningful if you throw away all sense of reality. Let's see,...


## Structure and data quality

### Display basic data: shapes, missing values, duplicates

In [5]:
build_dataset_overview({"train": train, "test": test})

,rows,columns,missing_values,duplicate_ids,duplicate_reviews
dataset,,,,,
train,25000,3,0,0,96
test,25000,2,0,0,199


### Display some examples of duplicate entries
- 96 duplicates = 96 copies of the exact same movie review text string.
- 96 copies above and beyond the original movie review text string(s) from which they are duplicated
- we don't know if it's:
- 1 original copy and 96 clones OR
- 96 originals which each have been duplicated once
- we just know there are 96 copies/duplicates

for our strategy, we will take the original copy and remove the dupes.

In [6]:
print("Two exact duplicate-review groups from the training data:")
display(duplicate_review_examples(train))

print("Two exact duplicate-review groups from the test data:")
display(duplicate_review_examples(test))

Two exact duplicate-review groups from the training data:


,duplicate_group,id,sentiment,review_preview
37,1,6827_4,0,"Dumb is as dumb does, in this thoroughly unint..."
5536,1,7645_4,0,"Dumb is as dumb does, in this thoroughly unint..."
7336,2,5088_1,0,"This guy has no idea of cinema. Okay, it seems..."
326,2,6645_1,0,"This guy has no idea of cinema. Okay, it seems..."


Two exact duplicate-review groups from the test data:


,duplicate_group,id,review_preview
18170,1,6419_7,Footprints is a very interesting movie that is...
45,1,6427_7,Footprints is a very interesting movie that is...
127,2,12380_7,The problem with so many people watching this ...
13430,2,7397_7,The problem with so many people watching this ...


## Positive and Negative Sentiment: Counts and Percentage

In [7]:
label_counts = train["sentiment"].value_counts().sort_index().rename("count").to_frame()
label_counts["percent"] = (100 * label_counts["count"] / len(train)).round(2)
label_counts.index.name = "sentiment"
label_counts

,count,percent
sentiment,,
0,12500,50.0
1,12500,50.0


## Review lengths

Character and word counts give an early indication of sequence-length requirements and unusually short or long reviews.

In [8]:
review_lengths = pd.DataFrame(
    {
        "train_characters": train["review"].str.len(),
        "train_words": train["review"].str.split().str.len(),
        "test_characters": test["review"].str.len(),
        "test_words": test["review"].str.split().str.len(),
    }
)
review_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1)

,train_characters,train_words,test_characters,test_words
count,25000.0,25000.0,25000.0,25000.0
mean,1327.7,233.8,1296.4,228.5
std,1005.2,173.7,978.1,168.9
min,52.0,10.0,32.0,4.0
50%,981.0,174.0,964.5,172.0
90%,2624.1,458.0,2549.0,444.0
95%,3440.2,598.0,3341.0,582.0
99%,5220.0,913.0,5157.1,901.0
max,13708.0,2470.0,13004.0,2278.0


In [9]:
longest_reviews = (
    train.assign(word_count=review_lengths["train_words"])
    .nlargest(5, "word_count")
    .loc[:, ["id", "sentiment", "word_count", "review"]]
)
display(longest_reviews)

,id,sentiment,word_count,review
3485,1175_9,1,2470,Match 1: Tag Team Table Match Bubba Ray and Sp...
20982,10044_9,1,1839,Titanic directed by James Cameron presents a f...
20347,4383_9,1,1830,**Attention Spoilers**<br /><br />First of all...
10503,4076_10,1,1723,By now you've probably heard a bit about the n...
4233,2662_10,1,1601,*!!- SPOILERS - !!*<br /><br />Before I begin ...


## Initial cleanup

Create cleaned copies for later modeling while preserving `train` and `test` as the untouched raw data. The shared `clean_reviews()` helper parses and removes HTML, normalizes whitespace, keeps the first occurrence of each review, and removes only later duplicates. Duplicate detection happens after HTML cleanup, so reviews that become identical after normalization are also deduplicated.

In [10]:
train_clean = clean_reviews(train)
test_clean = clean_reviews(test)

cleanup_report = build_cleanup_report(
    raw_datasets={"train": train, "test": test},
    cleaned_datasets={"train": train_clean, "test": test_clean},
)
cleanup_report

,raw_rows,clean_rows,removed_rows,remaining_duplicate_rows,remaining_angle_bracket_patterns
dataset,,,,,
train,25000,24903,97,0,1
test,25000,24799,201,0,6
